# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I'm using Random Forest with a Logistic Regression comparison. Random Forest fits this lane well because ML-04's leakage experiment already showed strong signal in features like impressions_30d, avg_position_30d, and position_volatility_30d — but the relationship isn't a single clean threshold (my ML-02/07 hand rule and simple tree comparisons showed unstable rankings depending on the metric), suggesting a joint, non-linear pattern across features that a tree-based ensemble can capture better than a single rule. I'm including Logistic Regression as a simpler baseline model alongside it, since the assignment rewards not over-complicating things if logistic regression matches Random Forest, that's worth reporting honestly rather than defaulting to the fancier model.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Client-holdout split (grouped by client_id), not a random row split. My question is "which pages should this content team prioritize" a model trained and tested on the same client's pages in both sets would look artificially strong, because it could memorize client-specific quirks rather than learning a generalizable pattern. ML-02's own experiment already showed this: my depth-2 tree scored 0.870 AUC on warehouse data, but the earlier in-sample comparison in Notebook 02 was optimistic before the honest holdout split. A client-holdout split is the only honest way to answer "would this work on a client we haven't seen."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarmad341/flyrank-internship-01"
REPO_DIR = "flyrank-internship-01"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Same features used in ML-07's leakage-safe set
features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count", "content_age_days"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

# My Week-4 baseline, scored on the SAME test set
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * df["impressions_90d"]
baseline_test_scores = df["baseline_score"].iloc[test_idx]

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(X_tr, y_tr)
rf_scores = rf.predict_proba(X_te)[:, 1]

# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
lr_scores = lr.predict_proba(X_te)[:, 1]

results = pd.DataFrame({
    "method": ["Week-4 Baseline (hand rule)", "Logistic Regression", "Random Forest"],
    "Precision@20": [
        precision_at_k(baseline_test_scores.values, y_te.values, 20),
        precision_at_k(lr_scores, y_te.values, 20),
        precision_at_k(rf_scores, y_te.values, 20),
    ],
    "Precision@50": [
        precision_at_k(baseline_test_scores.values, y_te.values, 50),
        precision_at_k(lr_scores, y_te.values, 50),
        precision_at_k(rf_scores, y_te.values, 50),
    ],
})
print(results)

                        method  Precision@20  Precision@50
0  Week-4 Baseline (hand rule)          0.50          0.62
1          Logistic Regression          0.65          0.66
2                Random Forest          0.75          0.72


On the same client-holdout test set, Random Forest achieved the best result at both Precision@20 (0.75) and Precision@50 (0.72), compared to the Week-4 baseline's 0.50 and 0.62. Logistic Regression also beat the baseline (0.65 / 0.66) but fell short of Random Forest. This shows genuine additional signal exists beyond the simple stale×visible rulethe gap is consistent across both K values, not just a lucky top-20, which makes it a credible improvement rather than noise. Unlike the earlier in-sample comparisons in Notebook 02 (where the tree's advantage disappeared on holdout), this result holds up under the same honest client-holdout validation, which is what makes it trustworthy enough to report.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf_scores
top50_model = test_df.sort_values("rf_score", ascending=False).head(50)
false_positives = top50_model[top50_model["is_declining_label"] == 0]
print(f"\n{len(false_positives)} of the top 50 model picks were false positives")
false_positives[features + ["is_declining_label"]].head(5)

impressions_90d           0.278719
avg_position              0.247534
content_age_days          0.159852
word_count                0.155721
ctr                       0.120867
days_since_last_update    0.037308
dtype: float64

16 of the top 50 model picks were false positives


,impressions_90d,days_since_last_update,avg_position,ctr,word_count,content_age_days,is_declining_label
22526,3445,104,39.0,0.09,1480.0,280,0
5399,1152,104,34.8,0.17,1535.0,280,0
24079,2573,13,13.4,0.58,2593.0,228,0
13529,3214,104,50.9,0.03,1599.0,280,0
11433,2292,13,22.0,0.35,2716.0,277,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.